In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

base_dir = Path("../../../variation_in_wind_drift_relation")

paths = [
    base_dir / "train_daily_alpha_theta.csv",
    base_dir / "val_daily_alpha_theta.csv",
    base_dir / "test_daily_alpha_theta.csv",
]

df = pd.concat(
    [pd.read_csv(path, parse_dates=["date"]) for path in paths],
    ignore_index=True,
)

df = df.sort_values("date").reset_index(drop=True)

fig, ax1 = plt.subplots(figsize=(12, 5))

# Left y-axis: alpha
ax1.scatter(df["date"], df["alpha"], s=12, color="red", label="alpha")
ax1.set_xlabel("Date")
ax1.set_ylabel("alpha", color="red")
ax1.tick_params(axis="y", labelcolor="red")
ax1.set_ylim(0, 0.1)

# Right y-axis: theta / beta
ax2 = ax1.twinx()
ax2.scatter(df["date"], df["theta_deg"], s=12, color="blue", label="theta_deg")
ax2.set_ylabel("theta_deg", color="blue")
ax2.tick_params(axis="y", labelcolor="blue")
ax2.set_ylim(-180, 180)

plt.title("Daily alpha and theta over time")

# Combined legend
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, loc="best")

fig.tight_layout()
plt.show()

In [ ]:
len(df)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

base_dir = Path("../../../variation_in_wind_drift_relation")
paths = [
    base_dir / "train_daily_alpha_theta.csv",
    base_dir / "val_daily_alpha_theta.csv",
    base_dir / "test_daily_alpha_theta.csv",
]

df = pd.concat([pd.read_csv(p, parse_dates=["date"]) for p in paths], ignore_index=True)
df = df.sort_values("date").reset_index(drop=True)

# If duplicate dates exist, average them first.
# Remove this groupby if you are certain each date occurs only once.
df_daily = (
    df.groupby("date", as_index=False)
      .agg({"alpha": "mean", "theta_rad": "mean"})
      .sort_values("date")
)

df_indexed = df_daily.set_index("date")
date_set = set(df_indexed.index)

def lag_k_autocorr_noncontinuous(df_idx, date_set, k):
    """
    For lag k, collect pairs (t, t+k) whenever both dates are present.
    Intermediate days do not need to be present.

    Returns:
        n_pairs: number of valid date pairs
        alpha_r: Pearson autocorrelation for alpha
        theta_circ_r: circular autocorrelation for theta, computed as mean cos(delta theta)
    """
    lag = pd.Timedelta(days=k)

    alpha_t, alpha_tk = [], []
    theta_t, theta_tk = [], []

    for d in date_set:
        d_k = d + lag

        if d_k in date_set:
            alpha_t.append(df_idx.loc[d, "alpha"])
            alpha_tk.append(df_idx.loc[d_k, "alpha"])

            theta_t.append(df_idx.loc[d, "theta_rad"])
            theta_tk.append(df_idx.loc[d_k, "theta_rad"])

    alpha_t = np.asarray(alpha_t)
    alpha_tk = np.asarray(alpha_tk)
    theta_t = np.asarray(theta_t)
    theta_tk = np.asarray(theta_tk)

    if len(alpha_t) < 2:
        return len(alpha_t), np.nan, np.nan

    alpha_r = np.corrcoef(alpha_t, alpha_tk)[0, 1]

    dtheta = np.angle(np.exp(1j * (theta_tk - theta_t)))
    theta_circ_r = np.mean(np.cos(dtheta))

    return len(alpha_t), alpha_r, theta_circ_r


print(f"{'Lag':>4}  {'n_pairs':>8}  {'alpha_r':>10}  {'theta_circ_r':>14}")
for k in range(1, 11):
    n, alpha_r, theta_r = lag_k_autocorr_noncontinuous(df_indexed, date_set, k)
    print(f"{k:>4}  {n:>8}  {alpha_r:>10.4f}  {theta_r:>14.4f}")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import spearmanr

base_dir = Path("/lustre/storeB/users/alessioc/Masters_Thesis/variation_in_wind_drift_relation")
paths = [
    base_dir / "train_daily_alpha_theta.csv",
    base_dir / "val_daily_alpha_theta.csv",
    base_dir / "test_daily_alpha_theta.csv",
]

df = pd.concat([pd.read_csv(p, parse_dates=["date"]) for p in paths], ignore_index=True)
df = df.sort_values("date").reset_index(drop=True)

# If duplicate dates exist, average them first.
df_daily = (
    df.groupby("date", as_index=False)
      .agg({"alpha": "mean", "theta_rad": "mean"})
      .sort_values("date")
)

df_indexed = df_daily.set_index("date")
date_set = set(df_indexed.index)

def lag_k_spearman_noncontinuous(df_idx, date_set, k):
    """
    For lag k, collect pairs (t, t+k) whenever both dates are present.
    Intermediate days do not need to be present.

    Returns:
        n_pairs: number of valid date pairs
        alpha_s: Spearman rank autocorrelation for alpha
        theta_persistence: angular persistence for theta, computed as mean cos(delta theta)
    """
    lag = pd.Timedelta(days=k)

    alpha_t, alpha_tk = [], []
    theta_t, theta_tk = [], []

    for d in date_set:
        d_k = d + lag

        if d_k in date_set:
            alpha_t.append(df_idx.loc[d, "alpha"])
            alpha_tk.append(df_idx.loc[d_k, "alpha"])

            theta_t.append(df_idx.loc[d, "theta_rad"])
            theta_tk.append(df_idx.loc[d_k, "theta_rad"])

    alpha_t = np.asarray(alpha_t)
    alpha_tk = np.asarray(alpha_tk)
    theta_t = np.asarray(theta_t)
    theta_tk = np.asarray(theta_tk)

    if len(alpha_t) < 2:
        return len(alpha_t), np.nan, np.nan

    alpha_s, _ = spearmanr(alpha_t, alpha_tk)

    dtheta = np.angle(np.exp(1j * (theta_tk - theta_t)))
    theta_persistence = np.mean(np.cos(dtheta))

    return len(alpha_t), alpha_s, theta_persistence


print(f"{'Lag':>4}  {'n_pairs':>8}  {'alpha_s':>10}  {'theta_persist':>14}")
for k in range(1, 11):
    n, alpha_s, theta_r = lag_k_spearman_noncontinuous(df_indexed, date_set, k)
    print(f"{k:>4}  {n:>8}  {alpha_s:>10.4f}  {theta_r:>14.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

lags = np.arange(1, 11)
alpha_rs = []
theta_rs = []

for k in lags:
    n, alpha_r, theta_r = lag_k_autocorr_noncontinuous(df_indexed, date_set, k)
    alpha_rs.append(alpha_r)
    theta_rs.append(theta_r)

fig, ax1 = plt.subplots(figsize=(8, 4))

ax1.plot(lags, alpha_rs, "o-", color="red", label="alpha_r (Pearson)")
ax1.set_xlabel("Lag (days)")
ax1.set_ylabel("alpha Pearson r", color="red")
ax1.tick_params(axis="y", labelcolor="red")
ax1.set_ylim(0, 0.6)

ax2 = ax1.twinx()
ax2.plot(lags, theta_rs, "s--", color="blue", label="theta_circ_r (mean cos Δθ)")
ax2.set_ylabel("theta mean cos(Δθ)", color="blue")
ax2.tick_params(axis="y", labelcolor="blue")
ax2.set_ylim(0.85, 0.95)

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, loc="upper right")

ax1.set_xticks(lags)
plt.title("Lag autocorrelation: alpha (Pearson) and theta (circular)")
fig.tight_layout()
plt.show()

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

TEXTWIDTH_PT = 418.25368
TEXTWIDTH_IN = TEXTWIDTH_PT / 72.27


def setup_pub_style(fontsize=9):
    mpl.rcParams.update({
        "font.size": fontsize,
        "axes.titlesize": fontsize,
        "axes.labelsize": fontsize,
        "xtick.labelsize": fontsize - 1,
        "ytick.labelsize": fontsize - 1,
        "legend.fontsize": fontsize - 1,
        "figure.dpi": 300,
        "savefig.dpi": 300,
    })


def fig_textwidth(height_ratio=0.4):
    return (TEXTWIDTH_IN, TEXTWIDTH_IN * height_ratio)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

setup_pub_style(fontsize=9)
sns.set_style("whitegrid")

alpha_color = "#1f77b4"
theta_color = "#d62728"

base_dir = Path("/lustre/storeB/users/alessioc/Masters_Thesis/variation_in_wind_drift_relation")
paths = [
    base_dir / "train_daily_alpha_theta.csv",
    base_dir / "val_daily_alpha_theta.csv",
    base_dir / "test_daily_alpha_theta.csv",
]
df = pd.concat(
    [pd.read_csv(path, parse_dates=["date"]) for path in paths],
    ignore_index=True,
)
df = df.sort_values("date").reset_index(drop=True)

fig, ax1 = plt.subplots(figsize=fig_textwidth(0.45))
ax2 = ax1.twinx()

ax1.scatter(df["date"], df["alpha"], s=16, color=alpha_color, label=r"$\alpha$", rasterized=True,
            edgecolors="black", linewidths=0.2, marker="^")
ax1.set_xlabel("Date")
ax1.set_ylabel(r"$\alpha$", color=alpha_color)
ax1.tick_params(axis="y", colors=alpha_color)
ax1.set_ylim(0, 0.1)
ax1.set_yticks([0, 0.025, 0.050, 0.075, 0.100])
ax1.grid(True, which="major", alpha=0.3)

ax2.scatter(df["date"], df["theta_deg"], s=16, color=theta_color, label=r"$\theta$ [$^\circ$]", rasterized=True,
            edgecolors="black", linewidths=0.2)
ax2.set_ylabel(r"$\theta$ [$^\circ$]", color=theta_color)
ax2.tick_params(axis="y", colors=theta_color)
ax2.set_ylim(-180, 180)
ax2.set_yticks([-180, -90, 0, 90, 180])
ax2.grid(False)

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles2 + handles1, labels2 + labels1, loc="upper right", frameon=True)

sns.despine(ax=ax1, right=False)
sns.despine(ax=ax2, left=False)

# plt.title(r"Daily $\alpha$ and $\theta$ over time")
fig.tight_layout()
plt.savefig("daily_alpha_theta_variations.pdf", dpi=300, bbox_inches="tight")  # Save the figure
plt.show()

In [ ]:
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

setup_pub_style(fontsize=9)
sns.set_style("whitegrid")

alpha_color = "#1f77b4"
theta_color = "#d62728"

base_dir = Path("/lustre/storeB/users/alessioc/Masters_Thesis/variation_in_wind_drift_relation")
paths = [
    base_dir / "train_daily_alpha_theta.csv",
    base_dir / "val_daily_alpha_theta.csv",
    base_dir / "test_daily_alpha_theta.csv",
]
df = pd.concat(
    [pd.read_csv(path, parse_dates=["date"]) for path in paths],
    ignore_index=True,
)
df = df.sort_values("date").reset_index(drop=True)

fig, ax1 = plt.subplots(figsize=fig_textwidth(0.4))
ax2 = ax1.twinx()

ax1.plot(lags, alpha_rs, "^-", color=alpha_color, linewidth=1.2, markersize=4,
         label=r"$r_\alpha$")
ax1.set_xlabel("Lag (days)")
ax1.set_ylabel(r"$r_\alpha$", color=alpha_color)
ax1.tick_params(axis="y", colors=alpha_color)
ax1.set_ylim(0, 0.6)
ax1.set_yticks([0, 0.15, 0.30, 0.45, 0.60])
ax1.set_xticks(lags)
ax1.grid(True, which="major", alpha=0.3)

ax2.plot(lags, theta_rs, "o-", color=theta_color, linewidth=1.2, markersize=4,
         label=r"$r_\theta$")
ax2.set_ylabel(r"$r_\theta$", color=theta_color)
ax2.tick_params(axis="y", colors=theta_color)
ax2.set_ylim(0.85, 0.95)
ax2.set_yticks([0.850, 0.875, 0.900, 0.925, 0.950])
ax2.grid(False)

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles2 + handles1, labels2 + labels1, loc="upper right", frameon=True)

sns.despine(ax=ax1, right=False)
sns.despine(ax=ax2, left=False)

# plt.title(r"Lag autocorrelation: $\alpha$ (Pearson) and $\theta$ (circular)")
fig.tight_layout()
plt.savefig("lag_autocorrelation_alpha_theta.pdf", dpi=300, bbox_inches="tight")  # Save the figure
plt.show()